# Phase 2: Conditional Variational Autoencoder (cVAE)
This notebook defines a generative probabilistic model. It trains an Encoder to compress physical circuit variations into a Latent Space, and a Decoder to instantly generate infinite valid physical circuits from Gaussian noise, conditioned on target performances. It uses the frozen PINN as a 'Physics Loss' guide during training.


In [1]:
import os
import random
import warnings
import joblib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cpu


In [2]:
# ---------------------------
# 1. LOAD DATA & FROZEN PINN
# ---------------------------
file_path = '../Data/FINAL_4CLASSES.csv'
df = pd.read_csv(file_path, encoding='utf-8', engine='python')

column_mapping = {
    'Gain(db)': 'Gain(dB)', 'Gain': 'Gain(dB)', 'gain': 'Gain(dB)',
    'Bandwidth': 'Bandwidth(Hz)', 'bandwidth': 'Bandwidth(Hz)',
    'GBW': 'GBW(MHz)', 'gbw': 'GBW(MHz)',
    'Power': 'Power(uW)', 'power': 'Power(uW)',
    'PM': 'PM(degree)', 'PhaseMargin': 'PM(degree)',
    'GM': 'GM(dB)', 'PSRR': 'PSRR(dB)', 'SlewRate': 'SlewRate (V/us)', 
    'SlewRate(V/µs)': 'SlewRate (V/us)', 'CMRR': 'CMRR(dB)'
}
df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns}, inplace=True)

df['Idc(uA)'] = 130.0
df['Length(um)'] = 0.18
df['CL(pF)'] = 10.0
df['CC(pF)'] = 55.0

FEATURE_COLUMNS = ['Temperature(°)', 'W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)', 'Idc(uA)', 'Length(um)', 'CC(pF)', 'CL(pF)']
REGRESSION_TARGETS = ['Gain(dB)', 'Bandwidth(Hz)', 'GBW(MHz)', 'Power(uW)', 'PM(degree)', 'GM(dB)', 'PSRR(dB)', 'SlewRate (V/us)', 'CMRR(dB)']

scaler_X = joblib.load('scaler_X.pkl')
scaler_y_reg = joblib.load('scaler_y_reg.pkl')
le = joblib.load('label_encoder.pkl')

class PINN(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_reg_outputs, n_classes, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.regression_head = nn.Linear(prev_dim, n_reg_outputs)
        self.classification_head = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        shared = self.backbone(x)
        return self.regression_head(shared), self.classification_head(shared)

pinn = PINN(
    input_dim=len(FEATURE_COLUMNS),
    hidden_dims=[128, 128, 128, 128],
    n_reg_outputs=len(REGRESSION_TARGETS),
    n_classes=len(le.classes_), 
    dropout_rate=0.047,
).to(device)

pinn.load_state_dict(torch.load('pinn_model.pth', map_location=device))
pinn.eval()
for param in pinn.parameters():
    param.requires_grad = False
print("Loaded pre-trained PINN model.")


Loaded pre-trained PINN model.


## Generative Architecture & Data Preparation
We prepare loaders for generating widths conditioned on targets & fixed conditions.


In [3]:
# Setup Data for cVAE Training
X_raw = df[FEATURE_COLUMNS].fillna(df[FEATURE_COLUMNS].mean())
y_reg_raw = df[REGRESSION_TARGETS].fillna(df[REGRESSION_TARGETS].mean())

X_scaled = scaler_X.transform(X_raw)
y_reg_scaled = scaler_y_reg.transform(y_reg_raw)

# Latent space indices
width_names = ['W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)']
width_indices = [FEATURE_COLUMNS.index(col) for col in width_names]
fixed_indices = [FEATURE_COLUMNS.index(col) for col in FEATURE_COLUMNS if col not in width_names]

X_widths = X_scaled[:, width_indices]
X_fixed  = X_scaled[:, fixed_indices]

# Train data loader
widths_tensor = torch.tensor(X_widths, dtype=torch.float32).to(device)
fixed_tensor = torch.tensor(X_fixed, dtype=torch.float32).to(device)
targets_tensor = torch.tensor(y_reg_scaled, dtype=torch.float32).to(device)

dataset = TensorDataset(widths_tensor, fixed_tensor, targets_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# ---------------------------
# 3. cVAE ARCHITECTURE
# ---------------------------
class cVAE(nn.Module):
    def __init__(self, width_dim=5, fixed_dim=5, target_dim=9, latent_dim=3, hidden_dims=[64, 64]):
        super().__init__()
        self.latent_dim = latent_dim
        cond_dim = fixed_dim + target_dim
        
        # ENCODER: P( Latent | Widths, Conditions )
        enc_dim = width_dim + cond_dim
        enc_layers = []
        for h in hidden_dims:
            enc_layers.extend([nn.Linear(enc_dim, h), nn.ReLU()])
            enc_dim = h
        self.encoder = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(enc_dim, latent_dim)
        self.fc_var = nn.Linear(enc_dim, latent_dim)
        
        # DECODER: P( Widths | Latent, Conditions )
        dec_dim = latent_dim + cond_dim
        dec_layers = []
        for h in hidden_dims:
            dec_layers.extend([nn.Linear(dec_dim, h), nn.ReLU()])
            dec_dim = h
        dec_layers.append(nn.Linear(dec_dim, width_dim))
        self.decoder = nn.Sequential(*dec_layers)
        
    def encode(self, x, cond):
        h = self.encoder(torch.cat([x, cond], dim=-1))
        return self.fc_mu(h), self.fc_var(h)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z, cond):
        return self.decoder(torch.cat([z, cond], dim=-1))
        
    def forward(self, x, cond):
        mu, logvar = self.encode(x, cond)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, cond)
        return x_recon, mu, logvar

cvae = cVAE(latent_dim=3).to(device)
optimizer = optim.Adam(cvae.parameters(), lr=0.001)

def cvae_loss_function(recon_x, x, mu, logvar, recon_targets, actual_targets, physics_weight=1.0):
    BCE = F.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    # Physics Loss: Pass generated widths through the frozen PINN constraint
    PHYS = F.mse_loss(recon_targets, actual_targets, reduction='sum')
    return BCE + KLD + physics_weight * PHYS, BCE, KLD, PHYS


In [4]:
# ---------------------------
# 4. TRAINING LOOP
# ---------------------------
epochs = 80
print("Training cVAE Generator...")

for epoch in range(1, epochs + 1):
    cvae.train()
    train_loss = 0
    bce_loss = 0
    kld_loss = 0
    phys_loss = 0
    
    for batch_idx, (w, f_fixed, t_target) in enumerate(dataloader):
        optimizer.zero_grad()
        cond = torch.cat([f_fixed, t_target], dim=-1)
        
        recon_w, mu, logvar = cvae(w, cond)
        
        # Build full tensor for PINN exactly as format expects
        full_recon = torch.zeros((w.size(0), len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        full_recon[:, width_indices] = recon_w
        full_recon[:, fixed_indices] = f_fixed
        
        pred_targets, _ = pinn(full_recon)
        
        loss, bce, kld, phys = cvae_loss_function(recon_w, w, mu, logvar, pred_targets, t_target, physics_weight=5.0)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        bce_loss += bce.item()
        kld_loss += kld.item()
        phys_loss += phys.item()
        
    if epoch % 10 == 0 or epoch == 1:
        n = len(dataloader.dataset)
        print(f"Epoch {epoch:03d} | Total: {train_loss/n:.4f} | Recon(Width): {bce_loss/n:.4f} | KLD: {kld_loss/n:.4f} | Physics(Target): {phys_loss/n:.4f}")

torch.save(cvae.state_dict(), 'cvae_generator.pth')
print("Model saved.")


Training cVAE Generator...
Epoch 001 | Total: 7.1572 | Recon(Width): 1.5362 | KLD: 0.1454 | Physics(Target): 1.0951
Epoch 010 | Total: 0.2768 | Recon(Width): 0.0460 | KLD: 0.0001 | Physics(Target): 0.0462
Epoch 020 | Total: 0.2623 | Recon(Width): 0.0424 | KLD: 0.0000 | Physics(Target): 0.0440
Epoch 030 | Total: 0.2579 | Recon(Width): 0.0421 | KLD: 0.0000 | Physics(Target): 0.0432
Epoch 040 | Total: 0.2565 | Recon(Width): 0.0420 | KLD: 0.0000 | Physics(Target): 0.0429
Epoch 050 | Total: 0.2549 | Recon(Width): 0.0421 | KLD: 0.0000 | Physics(Target): 0.0426
Epoch 060 | Total: 0.2542 | Recon(Width): 0.0422 | KLD: 0.0000 | Physics(Target): 0.0424
Epoch 070 | Total: 0.2546 | Recon(Width): 0.0423 | KLD: 0.0000 | Physics(Target): 0.0425
Epoch 080 | Total: 0.2534 | Recon(Width): 0.0423 | KLD: 0.0000 | Physics(Target): 0.0422
Model saved.


## Inference: Generating Novel Circuits
Now we feed a singular performance target and completely random Latent Noise (`z`) to instantly generate dozens of viable structural dimensions.


In [5]:
# ---------------------------
# 5. INSTANT GENERATION INFERENCE
# ---------------------------
cvae.eval()

# Pick a completely random sample index from df
sample_idx = 1042
sample_row = df.iloc[sample_idx]

target_metrics = {col: sample_row[col] for col in REGRESSION_TARGETS}
operating_conditions = {
    'Temperature(°)': sample_row['Temperature(°)'], 'Idc(uA)': sample_row['Idc(uA)'], 
    'Length(um)': sample_row['Length(um)'], 'CC(pF)': sample_row['CC(pF)'], 'CL(pF)': sample_row['CL(pF)']
}
print(f"Target Performance Requirement:")
for k,v in target_metrics.items(): print(f" - {k}: {v:.2f}")

target_array = np.array([[target_metrics[col] for col in REGRESSION_TARGETS]])
target_scaled = scaler_y_reg.transform(target_array)
f_dict = {**operating_conditions, **{k: 0 for k in width_names}}
initial_scaled = scaler_X.transform(np.array([[f_dict[col] for col in FEATURE_COLUMNS]]))

cond_fixed = torch.tensor(initial_scaled[0, fixed_indices], dtype=torch.float32, device=device).unsqueeze(0)
cond_target = torch.tensor(target_scaled[0], dtype=torch.float32, device=device).unsqueeze(0)

# The generative step:
num_generations = 5
print(f"\nGeneratively Sampling {num_generations} Designs...\n")

mu_widths_1d = torch.tensor(scaler_X.mean_[width_indices], dtype=torch.float32, device=device)
std_widths_1d = torch.tensor(scaler_X.scale_[width_indices], dtype=torch.float32, device=device)

with torch.no_grad():
    for i in range(num_generations):
        # Sample pure standard gaussian noise
        z = torch.randn((1, cvae.latent_dim), device=device)
        cond = torch.cat([cond_fixed, cond_target], dim=-1)
        
        recon_w_scaled = cvae.decode(z, cond)
        
        # Verify
        full_recon = torch.zeros((1, len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        full_recon[:, width_indices] = recon_w_scaled
        full_recon[:, fixed_indices] = cond_fixed
        
        pred_eval, _ = pinn(full_recon)
        pred_unscaled = scaler_y_reg.inverse_transform(pred_eval.cpu().numpy())[0]
        
        widths_unscaled = (recon_w_scaled[0] * std_widths_1d + mu_widths_1d).cpu().numpy()
        
        print(f"--- GENERATED DESIGN #{i+1} ---")
        for j, col in enumerate(width_names):
            print(f" > {col}: {widths_unscaled[j]:.4f}")
            
        print(" -> Physics Validation (Error from Target): ", end="")
        errors = []
        for j, col in enumerate(REGRESSION_TARGETS):
            err = abs(pred_unscaled[j] - target_metrics[col]) / (abs(target_metrics[col])+1e-8) * 100
            if err > 0.01:
                errors.append(f"{col}: {err:.1f}%")
        print(", ".join(errors) if errors else "Perfect match!")
        print()


Target Performance Requirement:
 - Gain(dB): 53.26
 - Bandwidth(Hz): 27319.08
 - GBW(MHz): 54.12
 - Power(uW): 2110.39
 - PM(degree): 54.31
 - GM(dB): 5.85
 - PSRR(dB): 48.68
 - SlewRate (V/us): 6.55
 - CMRR(dB): 61.41

Generatively Sampling 5 Designs...

--- GENERATED DESIGN #1 ---
 > W12(um): 21.0373
 > W34(um): 30.0232
 > W58(um): 17.8705
 > W6(um): 30.2685
 > W7(um): 31.0158
 -> Physics Validation (Error from Target): Gain(dB): 0.1%, Bandwidth(Hz): 2.3%, GBW(MHz): 1.2%, Power(uW): 0.5%, PM(degree): 0.1%, GM(dB): 0.6%, PSRR(dB): 0.1%, SlewRate (V/us): 0.9%, CMRR(dB): 0.1%

--- GENERATED DESIGN #2 ---
 > W12(um): 21.0651
 > W34(um): 30.0267
 > W58(um): 17.8601
 > W6(um): 30.2787
 > W7(um): 30.9908
 -> Physics Validation (Error from Target): Gain(dB): 0.1%, Bandwidth(Hz): 2.3%, GBW(MHz): 1.2%, Power(uW): 0.5%, PM(degree): 0.0%, GM(dB): 0.5%, PSRR(dB): 0.1%, SlewRate (V/us): 0.9%, CMRR(dB): 0.1%

--- GENERATED DESIGN #3 ---
 > W12(um): 21.0215
 > W34(um): 30.0113
 > W58(um): 17.8714
 >